In [0]:
dbutils.widgets.removeAll()

In [0]:
from datetime import date
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text("catalogo", "catalog_ecobici_dev")
dbutils.widgets.text("fecha_inicio", "2026-07-01")
dbutils.widgets.text("fecha_fin", "2026-10-31")
dbutils.widgets.text("storageName", "adlsmartdataproy26")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
storageName = dbutils.widgets.get("storageName")
fecha_inicio = dbutils.widgets.get("fecha_inicio")
fecha_fin   = dbutils.widgets.get("fecha_fin")

In [0]:
tabla_bronze     = f"{catalogo}.bronze.clima_raw"
tabla_silver     = f"{catalogo}.silver.clima_hora"
tabla_cuarentena = f"{catalogo}.silver.clima_cuarentena"

In [0]:
TEMP_MIN, TEMP_MAX = -10.0, 45.0
LLUVIA_MIN_MM = 0.1          # desde cuántos mm por hora se considera que "llueve"

In [0]:
# Descripción de los códigos de clima WMO que usa Open-Meteo
CODIGOS_WMO = [
    (0, "Despejado"), (1, "Mayormente despejado"), (2, "Parcialmente nublado"), (3, "Nublado"),
    (45, "Niebla"), (48, "Niebla con escarcha"),
    (51, "Llovizna ligera"), (53, "Llovizna moderada"), (55, "Llovizna intensa"),
    (56, "Llovizna helada ligera"), (57, "Llovizna helada intensa"),
    (61, "Lluvia ligera"), (63, "Lluvia moderada"), (65, "Lluvia fuerte"),
    (66, "Lluvia helada ligera"), (67, "Lluvia helada fuerte"),
    (71, "Nieve ligera"), (73, "Nieve moderada"), (75, "Nieve fuerte"), (77, "Granos de nieve"),
    (80, "Chubascos ligeros"), (81, "Chubascos moderados"), (82, "Chubascos violentos"),
    (85, "Chubascos de nieve ligeros"), (86, "Chubascos de nieve fuertes"),
    (95, "Tormenta"), (96, "Tormenta con granizo ligero"), (99, "Tormenta con granizo fuerte"),
]

In [0]:
bronze = spark.table(tabla_bronze)
 
CAMPOS = ["time", "temperature_2m", "relative_humidity_2m", "precipitation", "wind_speed_10m", "weather_code"]
 
horas = (
    bronze
    .select(
        F.explode(F.arrays_zip(*[F.col(f"hourly.{c}").alias(c) for c in CAMPOS])).alias("h"),
        "_archivo_origen",
        "_fecha_ingesta",
    )
    .select(
        # La API entrega "2026-07-01T00:00"
        F.try_to_timestamp(F.regexp_replace(F.col("h.time"), "T", " "), F.lit("yyyy-MM-dd HH:mm")).alias("fecha_hora"),
        F.col("h.temperature_2m").cast("double").alias("temperatura_c"),
        F.col("h.relative_humidity_2m").cast("double").alias("humedad_pct"),
        F.col("h.precipitation").cast("double").alias("precipitacion_mm"),
        F.col("h.wind_speed_10m").cast("double").alias("viento_kmh"),
        F.col("h.weather_code").cast("int").alias("codigo_clima"),
        "_archivo_origen",
        "_fecha_ingesta",
    )
)

In [0]:
# Descripción del código de clima (los códigos que no estén en la lista quedan como "Desconocido")
codigos = spark.createDataFrame(CODIGOS_WMO, "codigo_clima int, descripcion_clima string")

In [0]:
horas = (
    horas.join(F.broadcast(codigos), "codigo_clima", "left")
    .withColumn("descripcion_clima", F.coalesce("descripcion_clima", F.lit("Desconocido")))
)

In [0]:
w = Window.partitionBy("fecha_hora").orderBy("_archivo_origen")

In [0]:
clasificado = (
    horas
    .withColumn("rn", F.row_number().over(w))
    .withColumn(
        "motivo_rechazo",
        F.when(F.col("fecha_hora").isNull(), "fecha_invalida")
         .when(
             F.col("temperatura_c").isNull() | F.col("humedad_pct").isNull()
             | F.col("precipitacion_mm").isNull() | F.col("viento_kmh").isNull()
             | F.col("codigo_clima").isNull(),
             "medicion_nula",
         )
         .when(
             ~F.col("temperatura_c").between(TEMP_MIN, TEMP_MAX)
             | ~F.col("humedad_pct").between(0, 100)
             | (F.col("precipitacion_mm") < 0)
             | (F.col("viento_kmh") < 0),
             "valor_fuera_de_rango",
         )
         .when(F.col("rn") > 1, "duplicado")
    )
)

In [0]:
validas = (
    clasificado
    .filter(F.col("motivo_rechazo").isNull())
    .select(
        "fecha_hora",
        F.to_date("fecha_hora").alias("fecha"),
        F.hour("fecha_hora").alias("hora"),
        "temperatura_c",
        "humedad_pct",
        "precipitacion_mm",
        "viento_kmh",
        "codigo_clima",
        "descripcion_clima",
        (F.col("precipitacion_mm") >= LLUVIA_MIN_MM).alias("llueve"),
        "_archivo_origen",
        "_fecha_ingesta",
    )
)

In [0]:
cuarentena = clasificado.filter(F.col("motivo_rechazo").isNotNull()).drop("rn")

In [0]:
(
    validas.write
    .format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(tabla_silver)
)

In [0]:
(
    cuarentena.write
    .format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(tabla_cuarentena)
)

In [0]:
n_bronce     = horas.count()
n_silver     = spark.table(tabla_silver).count()
n_cuarentena = spark.table(tabla_cuarentena).count()
 
print(f"Horas recibidas en Bronce: {n_bronce:,}")
print(f"Silver:                    {n_silver:,}")
print(f"Cuarentena:                {n_cuarentena:,}")
assert n_bronce == n_silver + n_cuarentena, "Se perdieron o duplicaron filas"

Horas recibidas en Bronce: 2,952
Silver:                    2,952
Cuarentena:                0


In [0]:
# Cada hora debe aparecer una sola vez en Silver (es la llave del cruce con los viajes)
repetidas = spark.table(tabla_silver).groupBy("fecha_hora").count().filter(F.col("count") > 1).count()
assert repetidas == 0, f"Hay {repetidas} horas repetidas en Silver"

In [0]:
dias = (date.fromisoformat(fecha_fin) - date.fromisoformat(fecha_inicio)).days + 1
esperadas = dias * 24
recibidas = (
    spark.table(tabla_silver).select("fecha_hora")
    .union(spark.table(tabla_cuarentena).select("fecha_hora"))
    .filter(F.col("fecha_hora").isNotNull())
    .distinct().count()
)
print(f"Horas esperadas del periodo: {esperadas:,} | Horas distintas recibidas: {recibidas:,}")
assert recibidas == esperadas, "Faltan o sobran horas respecto al periodo configurado"

Horas esperadas del periodo: 2,952 | Horas distintas recibidas: 2,952


In [0]:
# Motivos de rechazo (si no hay filas, todo pasó las reglas)
display(
    spark.table(tabla_cuarentena)
    .groupBy("motivo_rechazo").count().orderBy(F.desc("count"))
)

motivo_rechazo,count


In [0]:
# Resumen por año: horas, temperatura media, horas con lluvia
display(
    spark.table(tabla_silver)
    .groupBy(F.year("fecha").alias("anio"))
    .agg(
        F.count("*").alias("horas"),
        F.round(F.avg("temperatura_c"), 1).alias("temp_media_c"),
        F.sum(F.col("llueve").cast("int")).alias("horas_con_lluvia"),
    )
    .orderBy("anio")
)

anio,horas,temp_media_c,horas_con_lluvia
2026,2952,17.4,1299


In [0]:
display(spark.table(tabla_silver).orderBy("fecha_hora").limit(24))

fecha_hora,fecha,hora,temperatura_c,humedad_pct,precipitacion_mm,viento_kmh,codigo_clima,descripcion_clima,llueve,_archivo_origen,_fecha_ingesta
2026-07-01T00:00:00Z,2026-07-01,0,14.9,88.0,0.0,1.7,3,Nublado,false,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/clima/clima_2026/part-00000-tid-6483016446650617859-0f8fc55e-b372-4171-9c68-8303b772e508-122-1-c000.txt,2026-11-12T23:34:17.7732Z
2026-07-01T01:00:00Z,2026-07-01,1,14.5,90.0,0.0,0.6,3,Nublado,false,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/clima/clima_2026/part-00000-tid-6483016446650617859-0f8fc55e-b372-4171-9c68-8303b772e508-122-1-c000.txt,2026-11-12T23:34:17.7732Z
2026-07-01T02:00:00Z,2026-07-01,2,14.2,90.0,0.0,1.2,3,Nublado,false,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/clima/clima_2026/part-00000-tid-6483016446650617859-0f8fc55e-b372-4171-9c68-8303b772e508-122-1-c000.txt,2026-11-12T23:34:17.7732Z
2026-07-01T03:00:00Z,2026-07-01,3,13.9,90.0,0.0,1.8,3,Nublado,false,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/clima/clima_2026/part-00000-tid-6483016446650617859-0f8fc55e-b372-4171-9c68-8303b772e508-122-1-c000.txt,2026-11-12T23:34:17.7732Z
2026-07-01T04:00:00Z,2026-07-01,4,13.8,90.0,0.0,1.3,3,Nublado,false,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/clima/clima_2026/part-00000-tid-6483016446650617859-0f8fc55e-b372-4171-9c68-8303b772e508-122-1-c000.txt,2026-11-12T23:34:17.7732Z
2026-07-01T05:00:00Z,2026-07-01,5,13.6,92.0,0.0,0.0,3,Nublado,false,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/clima/clima_2026/part-00000-tid-6483016446650617859-0f8fc55e-b372-4171-9c68-8303b772e508-122-1-c000.txt,2026-11-12T23:34:17.7732Z
2026-07-01T06:00:00Z,2026-07-01,6,13.7,93.0,0.0,1.3,3,Nublado,false,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/clima/clima_2026/part-00000-tid-6483016446650617859-0f8fc55e-b372-4171-9c68-8303b772e508-122-1-c000.txt,2026-11-12T23:34:17.7732Z
2026-07-01T07:00:00Z,2026-07-01,7,13.6,94.0,0.0,2.7,3,Nublado,false,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/clima/clima_2026/part-00000-tid-6483016446650617859-0f8fc55e-b372-4171-9c68-8303b772e508-122-1-c000.txt,2026-11-12T23:34:17.7732Z
2026-07-01T08:00:00Z,2026-07-01,8,14.6,87.0,0.0,2.1,3,Nublado,false,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/clima/clima_2026/part-00000-tid-6483016446650617859-0f8fc55e-b372-4171-9c68-8303b772e508-122-1-c000.txt,2026-11-12T23:34:17.7732Z
2026-07-01T09:00:00Z,2026-07-01,9,15.9,79.0,0.0,0.9,3,Nublado,false,abfss://raw@adlsmartdataproy26.dfs.core.windows.net/datasets/clima/clima_2026/part-00000-tid-6483016446650617859-0f8fc55e-b372-4171-9c68-8303b772e508-122-1-c000.txt,2026-11-12T23:34:17.7732Z
